# Bronze Layer — Raw Data Ingestion

**Databricks Pattern:** Medallion Architecture — Bronze = raw data preserved exactly as received.

This notebook demonstrates:
- Reading raw CSV files into Delta Lake format
- Adding audit/lineage columns without modifying source data
- Inspecting Delta transaction logs
- Querying schema and metadata


In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.utils.spark_session import get_spark_session
from src.utils.paths import RAW_PATH, BRONZE_PATH

spark = get_spark_session()
print(f'Spark version: {spark.version}')

In [ ]:
# ── Step 1: Generate synthetic data (if not already done) ────────────────────
from src.data.generate_synthetic import main as generate
generate()

In [ ]:
# ── Step 2: Inspect raw CSV before ingestion ─────────────────────────────────
import pandas as pd

raw_orders = pd.read_csv(str(RAW_PATH / 'orders.csv'))
print(f'Raw orders shape: {raw_orders.shape}')
print(f'Anomalies injected: {raw_orders["is_anomaly_injected"].sum()}')
raw_orders.head()

In [ ]:
# ── Step 3: Run Bronze pipeline ───────────────────────────────────────────────
from src.pipelines.bronze import main as run_bronze
run_bronze()

In [ ]:
# ── Step 4: Read back from Delta and inspect ──────────────────────────────────
bronze_orders = spark.read.format('delta').load(str(BRONZE_PATH / 'orders'))
print(f'Bronze orders: {bronze_orders.count():,} rows')
print(f'Schema:')
bronze_orders.printSchema()

In [ ]:
# ── Step 5: View audit columns added by Bronze pipeline ──────────────────────
bronze_orders.select(
    'order_id', '_bronze_ingested_at', '_source_file', '_batch_id', '_pipeline_version'
).show(5, truncate=False)

In [ ]:
# ── Step 6: View Delta transaction log ───────────────────────────────────────
from pathlib import Path
import json

log_dir = BRONZE_PATH / 'orders' / '_delta_log'
log_files = sorted(log_dir.glob('*.json'))
print(f'Delta log files: {len(log_files)}')

if log_files:
    with open(log_files[0]) as f:
        for line in f.readlines()[:3]:
            entry = json.loads(line)
            print(json.dumps(entry, indent=2)[:500])
            print('---')

In [ ]:
# ── Step 7: Anomaly preview in raw data ──────────────────────────────────────
from pyspark.sql import functions as F

bronze_orders.filter(F.col('is_anomaly_injected') == True).select(
    'order_id', 'total_amount', 'order_timestamp', 'anomaly_type'
).show(10, truncate=False)